<a href="https://colab.research.google.com/github/xzhen-hub/ML_Linear-Factor-Model/blob/main/g1_served_scurve_walkthrough_20261104.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Served S-curve and G1 CPR comparison: implementation walkthrough
Prepared November4,2026. Source commit: `47a64d418821f3aeb7b3585078247e09b1a5de92`.

**Goal:** establish whether our exact saved expanded Fannie and Freddie payoff models improve the CPR actually produced by the serving recipe, after distillation, on the same cells/pools and a confirmed realized basis. Existing monthly full-payoff tests do not answer that question.

**Readiness:** we have the saved trees/encoders, official builder/grader source, the owner's reference grades, and the exact June production-lineage comparator recipes. We do **not** yet have distilled artifacts built from our local expanded trees. A scoring callback is supplied, but integrating it into the owner's builder is new work. The notebook clearly separates runnable preparation from that integration and from the subsequent tests. Do not substitute the owner's ls2/lagls artifacts for our candidates.

**Important correction:** `src/mbs/inference/cpr_basis.py` defines served CPR as `model_total`: refinancing+turnover+curtailment+servicer buyouts, excluding scheduled amortization. My earlier description of the final production output as necessarily voluntary CPR was incomplete. The legacy grader reads `mv_cpr_truth`; its physical BigQuery table has no explicit basis column/description. Confirm its upstream definition and the official G1 benchmark basis with the owner. Keep a legacy-grader reproduction separate from a verified like-for-like total CPR evaluation. Do not silently call total CPR voluntary CPR.

**First fair window:** June2026 anchor/calibration, July-August2026 activity. Factor truth is August-September2026. A September-anchored production surface is lookahead for July-August; do not enable `--allow-lookahead`. The owner's actual grade records explicitly refuse that incumbent and use June e lineage as comparator. September activity can be added only after reviewing the correct pool-factor truth; all-zero September loan labels in our old snapshot do not prove the separate pool truth is empty.

**Two different questions:** (1) same-anchor new versus old tree under the same recipe form; (2) whether direct-tree gains survive distillation and final serving. Match all other settings: June rows, representative-loan samples/weights, calibration procedure/population, common curtailment model/factor/cbar, feature version, coupon keys, rate bridge, curve basis, story routing, fallback and grid/interpolation rules.

No new bucket or production-pointer change is required. Preparation and local grading need local files and existing read access. Any temporary BQ output created for the new builder must be uniquely named under `oasive_quant_sandbox`. Owner registry staging is a separate optional/necessary step for formal source-window admission, never done automatically here.


## Step1 — Open a NEW Colab notebook and restore this implementation kit
Upload `g1_served_scurve_implementation_20261104.zip` through the Files sidebar. Save this notebook in Google Drive. Old accuracy notebooks and session variables are not used.

The kit includes the pinned serving source ZIP, exact June comparator headers, four owner reference grade records, helper scripts, explicit input templates and read-only provenance. It includes verified model-only scoring archives for both agencies, without loan-level training/evaluation data. These are not FULL experiment recovery packages. Candidate surfaces are not built or included. You do not need to re-upload the large FULL backups for this scorer.


In [1]:
from pathlib import Path
import json,zipfile,hashlib,sys,subprocess,os,shutil,importlib.util
ROOT=Path('/content/g1_served_scurve_20261104');ROOT.mkdir(exist_ok=True)
KIT=Path('/content/g1_served_scurve_implementation_20261104.zip')
assert KIT.is_file() and zipfile.is_zipfile(KIT),'Upload the complete implementation kit'
with zipfile.ZipFile(KIT) as z:
    hashes=json.loads(z.read('served_plan_hashes.json'))
    for name,digest in hashes.items():
        p=(ROOT/name).resolve();assert p.is_relative_to(ROOT.resolve())
        b=z.read(name);assert hashlib.sha256(b).hexdigest()==digest,name
        if p.exists():assert hashlib.sha256(p.read_bytes()).hexdigest()==digest,'Conflicting existing kit file: '+name
        else:p.parent.mkdir(parents=True,exist_ok=True);p.write_bytes(b)
CONTRACT=json.loads((ROOT/'run_contract_TEMPLATE.json').read_text())
print('Kit verified:',ROOT)
print('Candidate surfaces:',CONTRACT['surface_generation_status'])


Kit verified: /content/g1_served_scurve_20261104
Candidate surfaces: not built


## Step2 — Inspect the correct original comparators and owner evidence
The supplied staged-folder URLs returned zero objects under those exact prefixes. Do not search for nonexistent ls2 Parquet files there: these June arms are legacy flat recipes held in the registry, with `form_version` absent/null. The production v2 surface Parquet files are in `gs://oasive-quant-features/scurve_v2/...`, as the verified registry headers show.

The exact fair first comparators are:
- Fannie: `scurve-legacy-20260601e-fannie30`.
- Freddie: `scurve-legacy-20260601e-freddie30`.
Both trace to raw tree `v2-20261002` and anchor June2026. The kit already contains their exported headers; no need to ask the owner to resend them.

The owner's ls2 and lagls are distinct pooled retrains. Inspect their records to understand the format and reproduce the pipeline, not to claim results for our expanded models.


In [2]:
import pandas as pd
for agency in ['fannie','freddie']:
    blob=json.loads((ROOT/f'production_june_{agency}_recipe.json').read_text());h=blob['metrics']
    print(agency,blob['version'],'source:',h['source_model_version'],'anchor:',h['calibrated_as_of'],
          'form:',h.get('form_version','legacy'),'cohorts:',len(h['cohorts']))
rows=[]
for path in sorted((ROOT/'owner_grade_evidence').glob('*.json')):
    d=json.loads(path.read_text())
    for model in d['models']:
        arms=model['arms']
        rows.append(dict(file=path.name,agency=model['agency'],candidate=arms['candidate']['version'],
          comparator=arms['comparator']['version'],incumbent_refused=arms['incumbent'].get('refused'),
          floor=d['args']['min_frame_coverage'],candidate_MAE=model['pooled']['all_horizons']['candidate']['mae'],
          comparator_MAE=model['pooled']['all_horizons']['comparator']['mae']))
display(pd.DataFrame(rows))
print('These are owner ls2 reference grades, NOT new expanded-model grades.')


fannie scurve-legacy-20260601e-fannie30 source: v2-20261002 anchor: 2026-06-01 form: legacy cohorts: 345
freddie scurve-legacy-20260601e-freddie30 source: v2-20261002 anchor: 2026-06-01 form: legacy cohorts: 197


,file,agency,candidate,comparator,incumbent_refused,floor,candidate_MAE,comparator_MAE
0,conv_prepay_scurve_30yr__h3-ls2-vs-june-e-floo...,fannie,scurve-legacy-20260601ls2-fannie30,scurve-legacy-20260601e-fannie30,LOOKAHEAD,0.1,0.986516,0.970889
1,conv_prepay_scurve_30yr__h3-ls2-vs-june-e-unfl...,fannie,scurve-legacy-20260601ls2-fannie30,scurve-legacy-20260601e-fannie30,LOOKAHEAD,0.0,1.041079,1.020289
2,conv_prepay_scurve_30yr_freddie__h3-ls2-vs-jun...,freddie,scurve-legacy-20260601ls2-freddie30,scurve-legacy-20260601e-freddie30,LOOKAHEAD,0.1,1.039557,0.995478
3,conv_prepay_scurve_30yr_freddie__h3-ls2-vs-jun...,freddie,scurve-legacy-20260601ls2-freddie30,scurve-legacy-20260601e-freddie30,LOOKAHEAD,0.0,1.120959,1.084781


These are owner ls2 reference grades, NOT new expanded-model grades.


## Step3 — Fix the experiment design before building curves
Use four explicitly labelled arms, each agency separately:

| Arm | Payoff tree | Representation | Purpose |
|---|---|---|---|
| A | October v2-20261002 | Existing June e legacy flat | Reproduce owner's original convention |
| B | Exact saved local expanded tree | June legacy flat built through adapted scorer | B versus A: paired tree read only if anchor rows/frame/calibration recipe match |
| C | October v2-20261002 | June v2 surface, matching production form/settings | Same-anchor v2 control; requires a new paired build |
| D | Exact saved local expanded tree | June v2 surface, identical settings to C | D versus C: main served-tree comparison |

An existing June e recipe is an owner-reference arm, not automatic proof that its feature frame matches our candidate. If frame/sample provenance differs, rebuild the old legacy on the exact frozen candidate frame before attributing B versus A to the tree. D versus A combines a tree change and representation change; label it accordingly. Do not call that a split-only or tree-only effect. The September-anchored live surface remains an additional reference only on months it is eligible to forecast.

Start with July-August. Report floored0.10 and unfloored0.00 frame-coverage reads. A floor means sample/frame UPB relative to truth UPB, not loan-level probability clipping. Include all common covered cells plus an explicitly mapped G1 subset. Premium is the owner's6.0-7.5 security-coupon bucket; it is broader than just6.0/6.5 G1. Keep other groups visible.

Copy the contract template to `run_contract.json`. Leave unknown confirmations false. Obtain from the owner: official monthly G1 membership/key mapping, exact truth basis/numerator/denominator and factor alignment, full-agency June payoff calibration procedure, source feature vintage/availability, and intended stories/fallback/curve basis. These are definition and engineering inputs, not a request to change production permissions.


In [3]:
contract_path=ROOT/'run_contract.json'
if not contract_path.exists():contract_path.write_text(json.dumps(CONTRACT,indent=2))
# Change only after obtaining actual definitions/evidence. Do not mark confirmations true merely to pass a check.
print(json.dumps(json.loads(contract_path.read_text()),indent=2))


{
  "anchor": "2026-06-01",
  "evaluation_activity_months": [
    "2026-07-01",
    "2026-08-01"
  ],
  "expected_factor_months": [
    "2026-08-01",
    "2026-09-01"
  ],
  "source_model_cutoff": "2026-05-31",
  "source_windows": {
    "fannie": "2022-01-01..2026-06-01",
    "freddie": "2021-05-01..2026-06-01"
  },
  "model_cpr_basis": "model_total",
  "buyouts_included": true,
  "scheduled_amortization_included": false,
  "truth_basis": null,
  "truth_basis_confirmed": false,
  "scheduled_balance_weight_confirmed": false,
  "factor_month_is_activity_plus_one": true,
  "official_G1_membership_confirmed": false,
  "scope": "unconfirmed research scope",
  "surface_generation_status": "not built",
  "local_worker_status": "scoring callback supplied; not a completed production surface integration",
  "source_registry_holdout_status": "unregistered local candidates are not automatically official holdout PASS",
  "controls": {
    "hold_curtail_model_fixed": true,
    "hold_s_cur_and_cbar_f

## Step4 — Prepare an isolated serving/grading runtime
There must be TWO environments. The saved encoders were built with scikit-learn1.6.1 and XGBoost3.4.1; the owner's current project has different dependencies. Never load the saved encoder into the grading environment or silently upgrade its implementation. The scoring worker returns keyed raw probabilities through Parquet; the grader consumes finished recipes/surfaces.

The code below restores the pinned tracked source and installs the owner's locked runtime in its own venv. This is the full project environment and can require substantial downloads/disk space; an existing owner-approved grading environment can be used instead. It does not install into the model-scoring runtime. Authentication needs your existing BQ/GCS read access and IAM database identity; do not use shared writer passwords or Secret Manager fallback.


In [4]:
REPO=ROOT/'owner_source';REPO.mkdir(exist_ok=True)
with zipfile.ZipFile(ROOT/'owner_serving_source_47a64d4.zip') as z:
    for name in z.namelist():
        p=(REPO/name).resolve();assert p.is_relative_to(REPO.resolve())
        if name.endswith('/'):continue
        p.parent.mkdir(parents=True,exist_ok=True)
        if not p.exists():p.write_bytes(z.read(name))
assert (REPO/'scripts/mbs/verify/grade_conventional_scurve.py').is_file()
# Run only when ready to install the owner's runtime. No model is fitted.
INSTALL_GRADING_ENV=False
if INSTALL_GRADING_ENV:
    subprocess.run([sys.executable,'-m','pip','install','uv'],check=True)
    subprocess.run([sys.executable,'-m','uv','sync','--frozen','--no-dev'],cwd=REPO,check=True)
GRADE_PY=REPO/'.venv/bin/python'
print('Expected grader Python:',GRADE_PY,'exists:',GRADE_PY.is_file())


Expected grader Python: /content/g1_served_scurve_20261104/owner_source/.venv/bin/python exists: False


In [5]:
# Run after the grading environment exists and when ready for authenticated read-only grading.
from google.colab import auth
auth.authenticate_user()
GRADE_ENV=os.environ.copy()
GRADE_ENV['PYTHONPATH']=str(REPO)
GRADE_ENV['POSTGRES_DB']='oasive'
GRADE_ENV['SENTRY_DSN']=''
GRADE_ENV['SCURVE_V2_GRID_CACHE']='4'  # cache only; does not change predictions or grid
GRADE_ENV['SCURVE_V2_CACHE_DIR']=str(ROOT/'surface_cache')
IAM_DATABASE_USER=''  # Your existing IAM database email identity, not a password.
# Fill before executing an actual grade; the wrapper refuses shared-secret fallback.
if IAM_DATABASE_USER:GRADE_ENV['POSTGRES_IAM_USER']=IAM_DATABASE_USER
if GRADE_PY.is_file():
    subprocess.run([str(GRADE_PY),'-m','scripts.mbs.verify.grade_conventional_scurve','--help'],cwd=REPO,env=GRADE_ENV,check=True)


## Step5 — Build the missing local-model-to-surface bridge (new engineering work)
**This is the prerequisite that does not yet exist. The kit's scorer is one component, not a completed distillation pipeline.** Do not run Step7 expecting candidate headers/Parquet files to exist. The owner's builder loads a registered BQ source recipe and uses ML.PREDICT on both heads; a local .ubj cannot be passed to `--source-version`.

Provide the owner/implementer with this exact integration specification:
1. Reuse the official `calibrate_prepay_scurve.py` and `calibrate_tree_surface.py` sampling, feature override, calibration, header, refi-seasonality and export logic.
2. Separate `READ_DATASET=oasive_quant_bq` from `WRITE_DATASET=oasive_quant_sandbox`; fully qualify frozen source tables. Use unique run-prefixed sample/grid tables. Simply changing the stock `--dataset` is insufficient because it controls both reads and writes.
3. Freeze June anchor rows and the same representative-loan sample/stratum weights for both arms. Use the full eligible inference frame for anchoring/distillation, not the5% training population or its retained-negative rows. Use actual feature rows, not a fabricated average loan. Match or explicitly reconcile the frame used by the existing June e baseline; otherwise rebuild a paired old-tree control on the frozen candidate frame. Preserve feature vintage, month-start rules and historical availability metadata.
4. Introduce a payoff-scoring callback at the anchor, neutral and grid payoff passes. The production branch uses exact BQ v2-20261002. The new branch writes final transformed25-feature requests to local Parquet shards with unique `request_id`, calls `local_payoff_score.py` in the saved scoring environment, then joins raw outputs1:1 by request_id. The supplied callback never calibrates; the builder remains responsible for calibration once.
5. Hold the original production curtailment head, its raw-odds calibration and cbar fixed in both branches. Our expanded candidate has no newly trained curtailment head. Record that the test changes the payoff component.
6. Use the owner's actual per-agency June calibration population/procedure for the primary served test. Earlier G1-only June factors are a separately labelled research sensitivity; do not silently use them as full-agency production calibration. Distinguish raw factor s_pay from an extra factor applied after0.12 sampling correction.
7. Use official `cell_overrides`, `neutral_sql`/`_neutral_overrides`, `pay_turn_sql`, `curtail_sql`, `base_sql` semantics. Turnover is min(calibrated actual payoff, calibrated neutral payoff) PER LOAN before weighting. Refi=pay-turn. Total unscheduled SMM=pay+curtail; do not add turnover a second time or add scheduled amortization.
8. Preserve zero-offset identity even for rows outside history ties. Incentive shift also changes PMMS and10Y with the declared beta; rolling incentive is its own axis. Preserve null media/history handling and missing-history disposition.
9. For v2, match grid, interpolation, anchors, full12-lag histories/validity shares, rate bridge and curve basis. Apply production refi-seasonality flattening identically and record original versus flattened anchor deviation. Do not claim unflattened direct-tree parity for a deliberately flattened surface.
10. Export local production-compatible recipe headers and Parquet with the correct per-cohort row-group layout and SHA hashes. Local files avoid GCS writes. `--no-register` alone does not prevent GCS export; `--no-export` prevents export but does not magically create local Parquet. The stock builder therefore needs an explicit local-only export path and isolated sandbox destinations.
11. Preserve story declarations, admitted sets, priority hashes and fallback rules in the paired experiment. Tag any unchanged old-tree fallback exposure; do not count it as newly scored candidate exposure. Start all-only for engineering validation, then test the actual intended routing before claiming production equivalence.
12. Include exact expanded model SHA in `research_candidate_model_sha256` on candidate header and honest new source_model/source_model_version identifiers. Never label our local model as ls2, lagls or v2-20261002. If no source registry row exists, the official source-window holdout check remains unknown/false. An owner-staged source metadata row or an approved source-window extension is needed before calling it an official admission PASS.

Default grid:27 age ×12 calendar ×21 incentive ×9 rolling ×5 burnout ×9 media =2,755,620 states per cohort, before components/sample loans. At26 loans/cohort this is71,646,120 payoff evaluations per cohort. Do not launch full-portfolio or53 monthly full-grid builds without a cost/slot/disk plan. First smoke-test a small declared grid/cohort set and label it engineering-only; then use production grid and sampling-error validation for the deciding read. Cache/chunking changes are allowed without changing the mathematical experiment.

Expected deliverables per agency: new June flat recipe B; paired June old/new v2 headers C/D plus local Parquet; calibration/source manifest; anchor/sample parity report; coverage/fallback report; raw-scoring provenance; source-window metadata status. No results for these new surfaces are claimed in this notebook. The callback alone is insufficient to run the grading steps.


### Exact raw-scorer command for the implementer
Use the included fannie_saved_model_scoring_assets.zip and freddie_saved_model_scoring_assets.zip; each contains the exact raw model, encoder, matrix module and recorded versions selected from the verified FULL backup. Create a separate venv from each archive's recorded versions before running. It must match the saved NumPy/pandas/SciPy/sklearn/PyArrow/XGBoost versions; the helper refuses mismatches.

The input request shard has `request_id`, `agency` and all25 final transformed feature columns. It can contain repeated loans at different cells, but request_id must be unique. A shard is capped at one million rows; split without dropping any grid states. Output is request_id plus RAW payoff probability. The official builder calibrates exactly once afterward. A neutral request is scored through the same saved model on the official neutral-transformed features. Do not refit the encoder to remove unknown categories in the wider serving frame; record their exposure and treat a changed encoder as a different candidate.

```python
subprocess.run([
    '/content/model_score_env/bin/python',
    str(ROOT/'local_payoff_score.py'),
    '--agency','fannie',
    '--full-zip',str(ROOT/'fannie_saved_model_scoring_assets.zip'),
    '--input','/content/anchor_grid_requests_0001.parquet',
    '--output','/content/anchor_grid_raw_scores_0001.parquet'
],check=True)
```

For Freddie change agency and use `ROOT/freddie_saved_model_scoring_assets.zip`. The argument name remains `--full-zip` for compatibility; it accepts these verified selected-asset archives as well as the original FULL packages. These request files must be generated by the adapted builder; the command is an integration example, not a supplied populated input file.


### Set up the saved-model scoring environment when ready
This installs only the six recorded scoring packages in a separate environment. No runtime restart is needed because the worker runs in a fresh subprocess. The worker remains a callback: this does not generate surface requests or a candidate surface.


In [6]:
import venv
SCORE_ENV=Path('/content/model_score_env')
SCORE_PY=SCORE_ENV/'bin/python'
SETUP_SCORING_ENV=False
if SETUP_SCORING_ENV:
    with zipfile.ZipFile(ROOT/'fannie_saved_model_scoring_assets.zip') as z:
        versions=json.loads(z.read('recovered/expanded/manifest.json'))['versions']
    venv.EnvBuilder(with_pip=True).create(SCORE_ENV)
    requirements=[name+'=='+versions[name] for name in ['numpy','pandas','scipy','scikit-learn','pyarrow','xgboost']]
    subprocess.run([str(SCORE_PY),'-m','pip','install',*requirements],check=True)
    print('Scoring environment ready:',SCORE_PY)


## Step6 — Validate distillation before grading outcomes
Check at least: saved-model March replay; raw scorer parity; same sample request IDs/weights; all-anchor transformed inputs equal original rows; unflattened anchor components match direct scoring; flattened deviation explicitly recorded; no missing/duplicate grid states; finite SMM0..1; per-loan turn<=pay; correct history seed/ties; correct coupon convention; same treasury/SOFR basis and beta; local Parquet layout/SHA passes serving loader; fallback exposure reported; sample-versus-full-cohort error and off-grid direct-tree-versus-interpolated-surface deviations measured in CPR.

Use the official `gate_scurve_v2.py` on each v2 artifact. It tests serving shape/exactness/clamping, not out-of-time forecast accuracy. Do not use `--bridge-refused-export` to make a rejected artifact look production-compatible. Coordinate tolerance changes with the owner before seeing deciding results.

```python
subprocess.run([
 str(GRADE_PY),'-m','scripts.mbs.verify.gate_scurve_v2',
 '--recipe-file',str(ROOT/'artifacts/fannie_new_june_v2.json'),
 '--surface',str(ROOT/'artifacts/fannie_new_june_v2.parquet'),
 '--legacy-file',str(ROOT/'artifacts/fannie_new_june_flat.json'),
 '--scratch',str(ROOT/'gate_scratch/fannie')
],cwd=REPO,env=GRADE_ENV,check=True)
```

Run same check for old-tree June v2 and Freddie. Grid-gate success is necessary but does not establish better G1 CPR forecasts.


## Step7 — Grade paired local artifacts, June anchor to July-August
Run this only after Step5 actually produces the files and Step6 passes. The read-only wrapper changes only how the two local arms are loaded. It keeps the owner's rate path, interpolation, common-cell logic, source-window check, settled-truth guard, frame coverage and per-cell errors. It never overrides an unknown holdout status. The live production pointer is outside this explicit paired-source test; the baseline file must trace to v2-20261002.

The main call uses new v2 D against matched old v2 C. For the first legacy test, point to B/A headers, omit surface paths, and use their same legacy form. D versus A is allowed only with explicit `combined_tree_and_form_change` mode and must be interpreted as a combined pipeline change.


In [7]:
ART=ROOT/'artifacts'
READY_TO_GRADE=False  # Set only after genuine candidate builds/parity checks exist.
if READY_TO_GRADE:
    assert GRADE_PY.is_file() and GRADE_ENV.get('POSTGRES_IAM_USER'),'Existing grading runtime and IAM identity required'
    for agency in ['fannie','freddie']:
        for floor,label in [(0.1,'floored'),(0.0,'unfloored')]:
            files_needed=[ART/f'{agency}_new_june_v2.json',ART/f'{agency}_old_june_v2.json',
                          ART/f'{agency}_new_june_v2.parquet',ART/f'{agency}_old_june_v2.parquet']
            assert all(p.is_file() for p in files_needed),'Step5 candidate/paired baseline artifacts are not built'
            out=ROOT/'new_grades'/f'{agency}_new_vs_old_june_{label}.json'
            subprocess.run([str(GRADE_PY),str(ROOT/'grade_local_pair.py'),
                '--agency',agency,'--candidate-file',str(files_needed[0]),'--baseline-file',str(files_needed[1]),
                '--candidate-surface',str(files_needed[2]),'--baseline-surface',str(files_needed[3]),
                '--floor',str(floor),'--contract',str(contract_path),'--out',str(out)],
                cwd=REPO,env=GRADE_ENV,check=True)
else:
    print('Preparation only: no expanded-model S-curve grade has been run. Complete builder integration first.')


Preparation only: no expanded-model S-curve grade has been run. Complete builder integration first.


## Step8 — Inspect actual CPR predictions and grades
Each new grade must contain both candidate and comparator, same June anchor, source identities, graded July/August activity, August/September factor truth, no lookahead, common-cell coverage and explicit source-window holdout status. If source metadata is absent, the score can be a research diagnostic but is not a formal out-of-sample admission PASS. Do not substitute another source version to clear this.

The extractor generates cell-level actual CPR, comparator CPR, candidate CPR, signed errors and MAE; monthly weighted MAE/bias; and provenance. It does not call a mean of annual cell CPR the official aggregate G1 CPR. Validate the full eligible cell set/excluded exposure, not just the surviving winners.


In [8]:
if READY_TO_GRADE:
    for agency in ['fannie','freddie']:
        for label in ['floored','unfloored']:
            grade=ROOT/'new_grades'/f'{agency}_new_vs_old_june_{label}.json'
            out=ROOT/'tables'/f'{agency}_{label}'
            subprocess.run([str(GRADE_PY),str(ROOT/'extract_grade_tables.py'),
                '--grade',str(grade),'--out',str(out)],cwd=REPO,env=GRADE_ENV,check=True)
            display(pd.read_csv(out/'served_cpr_monthly_accuracy.csv'))
            display(pd.read_csv(out/'arm_provenance.csv'))


## Step9 — Define and report the G1 subset correctly
The owner grader is by security coupon × collateral vintage (and story when requested). Our earlier actual-pool adjusted-vintage research roster is not certified as the same population. Do not assume `vintage_year=2024,coupon6.0/6.5` equals the official G1 definition.

Obtain a confirmed cell mapping or monthly pool roster. For a confirmed cell-based target, fill owner_G1_cells_TEMPLATE.json with actual agency/coupon/vintage/story keys and pass `--cells` to the extractor. Empty list means missing scope, never zero error. Label a proxy explicitly if membership is not official.

For the actual served pool population, export each pool's production and candidate SMM from the normal serving lookup/path with identical pool inputs, date and scenario. This must preserve WAC key convention, story precedence/admission, history and fallback. Do not substitute raw loan payoff probability for pool served SMM. Export an independent risk-set roster and reconcile both arms to it before calculating results. The pool grader refuses missing/extra IDs or changed denominators.

Files required: paired_pool_export.csv and independent_pool_risk_set.csv, matching the supplied column templates. SMM is fractional0..1. The owner must confirm the total-unscheduled numerator, scheduled-balance denominator and factor-month alignment. A voluntary-only comparison requires an explicitly matched model decomposition/truth; the supplied total-CPR function does not implement that by relabelling buyouts away.


In [9]:
READY_POOL_EXPORTS=False
if READY_POOL_EXPORTS:
    subprocess.run([sys.executable,str(ROOT/'grade_pool_cpr.py'),
        '--input',str(ROOT/'paired_pool_export.csv'),
        '--risk-set',str(ROOT/'independent_pool_risk_set.csv'),
        '--contract',str(contract_path),'--out',str(ROOT/'G1_pool_results')],check=True)
    display(pd.read_csv(ROOT/'G1_pool_results/G1_served_CPR_monthly_comparison.csv'))
else:
    print('Pool-level serving exports and confirmed basis/membership are not supplied yet.')


Pool-level serving exports and confirmed basis/membership are not supplied yet.


## Step10 — Read the deciding tables and preserve weak groups
Required main table: activity month, agency, official/proxy G1 scope, actual CPR, October-lineage same-anchor CPR, new served CPR, both biases, both absolute errors, evaluated pool/cell count, UPB, coverage floor and fallback share. Report every month and both coupons, plus premium/discount/cusp and young-age buckets where covered.

Read floored0.10 as the owner convention and unfloored0.00 as coverage sensitivity. Same uncovered population must not disappear differently between arms. Summarize absolute errors before pooling so positive/negative cell errors cannot cancel. For combined portfolio CPR, use the owner's confirmed SMM/denominator method; do not average Fannie/Freddie percentage improvement or annual CPR values.

Keep prior Freddie delinquent6.5 and Fannie small6.5 loan-level diagnostics in a parallel report. The cohort-level official grader has no generic `--delinquent` or `--small-loan` switch; a single generic cohort curve may not preserve individual-loan distinctions. Do not claim those groups were graded merely by running the all-cell CPR script. Use confirmed pool stories/subpopulations or separate served/direct diagnostics as supported.

Interpretation: direct tree improves but served output does not -> investigate compression, representative sampling, anchor level, history/seasonality, routing/fallback and curtail contribution. Candidate improves pooled MAE but premium deteriorates -> not a broad owner-gate win. A gate pass is not a forecast win. A forecast win with unknown source/basis/membership is not promotion evidence. Apply the owner's actual rule, including recorded premium tolerance0.1CPR where applicable; do not invent universal thresholds for our candidate.


## Step11 — Add September and historical months only as separate, valid extensions
The owner pool grade records already have nonzero September CPR at October factor dates, while our loan snapshot September labels were zero. Check the pool truth's current completeness independently; the sources are different. If confirmed, extend June-anchor horizon to3 using the same frozen artifacts, without tuning on September.

To check whether March tree gains survive serving, build an explicitly matched March-anchor old/new recipe pair (or another valid chosen historical anchor), using the same June factors only as a clearly retrospective sensitivity. A June-anchored surface is not an as-known March forecast; do not backdate its header or enable lookahead to claim otherwise.

Full historical monthly surface comparison requires correctly re-anchored artifacts/snapshots and valid cohort coverage for each month, with known training/calibration lookahead labelled. Reusing a tree trained through May2026 on earlier months remains historical fit. Do not run53 enormous grids before confirming the two-month serving result and cost budget. Preserve Jan2022-May2026 calendar gaps rather than manufacture CPR for pools that did not exist.


## Step12 — Export evidence and prepare a promotion decision
Save candidate/baseline header+Parquet hashes, raw model/encoder/matrix hashes, calibration/scoring scales, source feature/label identities and availability, sample/coverage/fallback records, gate results, official grader JSON, factor alignment, G1 mapping, truth-basis confirmation and final tables. Exact raw scores, surface rows and served forecasts must have separate provenance so loss of accuracy can be located.

Download a FULL recovery package containing all local artifacts actually built and a small-results package with JSON/CSV/code. Store them under Y:/OASIVE/. Large surfaces can need multiple verified shards; do not create an incomplete ZIP or include a ZIP inside itself. Owner review and staging/promotion are separate actions. This notebook does not change registry status or production pointers.

## One message to the owner/implementer
We have exact saved local Fannie/Freddie25-feature payoff models and June production-lineage comparator headers. Please help integrate the supplied raw-scoring callback into the official legacy/v2 builders, preserving official sample/override/component/export logic, with source reads from oasive_quant_bq and unique outputs only in oasive_quant_sandbox/local files. Hold the October curtailment head/factors/cbar fixed. We need paired June old/new artifacts, full-agency calibration provenance, source-window metadata status, official G1 membership and the exact pool CPR truth basis/denominator. The first read will be July-August, floored/unfloored, followed by the official gate and served-pool comparison. No production switch or bucket creation is requested.

## What was verified while preparing this kit
Read-only object inventories:16 owner grade records found; zero objects under the supplied legacy recipe prefixes; production Parquet paths verified through registry. Exported actual June e comparator recipes:345 Fannie cohorts and197 Freddie cohorts. Read-only November4 registry showed production surface versions ending -sofr for both agencies. Source metadata confirms mv_cpr_truth is a physical table with no basis field/description. Inspected code matches the pinned commit. None of these checks built or graded our new expanded surfaces.

Helper files: local_payoff_score.py (raw callback, not full builder), grade_local_pair.py (read-only local-file arms using the official grader), extract_grade_tables.py (actual/candidate/baseline cell CPR tables), grade_pool_cpr.py (confirmed-risk-set total CPR comparison), run_contract_TEMPLATE.json and pool/cell templates. The source-window admission check is never overridden. New surface generation remains the outstanding implementation prerequisite.
